# 🍩 DeepSpringfield — Notebook d'Évaluation
## Évaluation du meilleur modèle YOLOv8 sur le jeu de test

Ce notebook charge le meilleur modèle issu du notebook d'entraînement
et l'évalue **uniquement sur le jeu de test** (jamais vu pendant l'entraînement).

## 1. Installation & Imports

In [ ]:
!pip install ultralytics --quiet

import os
import random
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from pathlib import Path
from PIL import Image
from matplotlib.lines import Line2D
from ultralytics import YOLO
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")

## 2. Chemins & Chargement du Modèle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_ROOT = Path("/content/drive/MyDrive/M2 SISE/DL/DeepSpringfield_ComputerVision/dataset")
DATA_YAML    = DATASET_ROOT / "data.yaml"
TEST_IMG     = DATASET_ROOT / "test" / "images"
TEST_LBL     = DATASET_ROOT / "test" / "labels"
MODELS_DIR   = Path("/content/drive/MyDrive/M2 SISE/DL/DeepSpringfield_ComputerVision/models")

with open(DATA_YAML) as f:
    config = yaml.safe_load(f)
CLASS_NAMES = config["names"]
NUM_CLASSES = config["nc"]

print(f"Classes    : {CLASS_NAMES}")
print(f"Nb classes : {NUM_CLASSES}")

BEST_PT = MODELS_DIR / "best_simpsons_model.pt"
assert BEST_PT.exists(), f"Modèle introuvable : {BEST_PT}"

model    = YOLO(str(BEST_PT))
DEVICE   = 0 if torch.cuda.is_available() else "cpu"
IMG_SIZE = 640

test_images = list(TEST_IMG.glob("*.jpg")) + list(TEST_IMG.glob("*.jpeg")) + list(TEST_IMG.glob("*.png"))

print(f"\n✅ Modèle chargé : {BEST_PT.name}")
print(f"   Device        : {DEVICE}")
print(f"   Images de test: {len(test_images)}")

In [ ]:
# Rappel du classement des expériences pour contexte
csv_path = MODELS_DIR.parent / "models" / "experiments_results.csv"
if not csv_path.exists():
    csv_path = MODELS_DIR / "experiments_results.csv"

if csv_path.exists():
    df_exp = pd.read_csv(csv_path).sort_values("mAP50_val", ascending=False)
    print("Rappel du classement des expériences (validation) :\n")
    cols = ["run", "model", "lr0", "optimizer", "augmentation", "mAP50_val", "mAP50_95_val"]
    print(df_exp[cols].to_string(index=False, float_format="{:.4f}".format))
else:
    print("[!] experiments_results.csv non trouvé")

## 3. Évaluation Quantitative sur le Jeu de Test

In [ ]:
print("="*55)
print("ÉVALUATION SUR LE JEU DE TEST")
print("="*55)

metrics_test = model.val(
    data    = str(DATA_YAML.resolve()),
    split   = "test",
    imgsz   = IMG_SIZE,
    device  = DEVICE,
    verbose = False,
)

map50   = metrics_test.box.map50
map5095 = metrics_test.box.map
prec    = metrics_test.box.mp
rec     = metrics_test.box.mr
f1      = 2 * prec * rec / (prec + rec + 1e-9)

print(f"\n  mAP@50      : {map50:.4f}")
print(f"  mAP@50-95   : {map5095:.4f}")
print(f"  Précision   : {prec:.4f}")
print(f"  Rappel      : {rec:.4f}")
print(f"  F1-score    : {f1:.4f}")

In [ ]:
# Métriques par classe
ap_idx   = metrics_test.box.ap_class_index
ap50_cls = metrics_test.box.ap50
ap_cls   = metrics_test.box.ap

df_classes = pd.DataFrame({
    "Classe"   : [CLASS_NAMES[i] for i in ap_idx],
    "AP@50"    : ap50_cls,
    "AP@50-95" : ap_cls,
})
mean_row = pd.DataFrame([{"Classe": "mAP (moyenne)", "AP@50": map50, "AP@50-95": map5095}])
df_classes = pd.concat([df_classes, mean_row], ignore_index=True)

print("Métriques par classe (set de test) :")
print("-"*45)
print(df_classes.to_string(index=False, float_format="{:.4f}".format))
df_classes.to_csv("test_metrics_per_class.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
class_data = df_classes[df_classes["Classe"] != "mAP (moyenne)"]
colors = sns.color_palette("Set2", len(class_data))
bars = ax.bar(class_data["Classe"], class_data["AP@50"], color=colors, edgecolor="black")
ax.axhline(y=map50, color="red", linestyle="--", linewidth=2, label=f"mAP@50 = {map50:.3f}")
ax.set_ylim(0, 1.05)
ax.set_title("AP@50 par classe — Jeu de test", fontsize=13)
ax.set_ylabel("AP@50")
ax.legend(fontsize=10)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.01, f"{h:.3f}",
            ha="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig("ap50_per_class_test.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Visualisations Qualitatives
### 4.1 Prédictions sur un échantillon du jeu de test

In [ ]:
random.seed(42)
sample = random.sample(test_images, min(9, len(test_images)))

preds = model.predict(
    source  = [str(p) for p in sample],
    imgsz   = IMG_SIZE,
    conf    = 0.25,
    iou     = 0.5,
    device  = DEVICE,
    save    = False,
    verbose = False,
)

cmap = plt.colormaps["tab10"]
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes = axes.flatten()

for ax, img_path, result in zip(axes, sample, preds):
    img = Image.open(img_path).convert("RGB")
    w_img, h_img = img.size
    ax.imshow(img)

    # Ground Truth (vert pointillé)
    lbl_path = TEST_LBL / (img_path.stem + ".txt")
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5: continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:])
                x1 = (cx - bw/2)*w_img; y1 = (cy - bh/2)*h_img
                ax.add_patch(patches.Rectangle((x1, y1), bw*w_img, bh*h_img,
                    linewidth=2, edgecolor="lime", facecolor="none", linestyle="--"))

    # Prédictions
    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            cls_id = int(box.cls[0].cpu())
            conf   = float(box.conf[0].cpu())
            color  = cmap(cls_id / 10)
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor=color, facecolor="none"))
            ax.text(x1, y1-4, f"{CLASS_NAMES[cls_id]} {conf:.2f}",
                    color="white", fontsize=8, fontweight="bold",
                    bbox=dict(facecolor=color, alpha=0.8, pad=1))

    ax.set_title(img_path.name[:35], fontsize=7)
    ax.axis("off")

for ax in axes[len(sample):]: ax.axis("off")

legend_elements = [
    Line2D([0],[0], color="lime", linewidth=2, linestyle="--", label="Ground Truth"),
    Line2D([0],[0], color="gray", linewidth=2, linestyle="-",  label="Prédiction"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=2, fontsize=11)
plt.suptitle("Prédictions vs Ground Truth — Jeu de test", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig("test_predictions_sample.png", dpi=150, bbox_inches="tight")
plt.show()


### 4.2 Cas réussis vs erreurs courantes

In [ ]:
def compute_iou(box_pred, box_gt):
    xA = max(box_pred[0], box_gt[0]); yA = max(box_pred[1], box_gt[1])
    xB = min(box_pred[2], box_gt[2]); yB = min(box_pred[3], box_gt[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    area_pred = (box_pred[2]-box_pred[0]) * (box_pred[3]-box_pred[1])
    area_gt   = (box_gt[2]-box_gt[0])     * (box_gt[3]-box_gt[1])
    union = area_pred + area_gt - inter + 1e-9
    return inter / union

def load_gt_boxes(lbl_path, w_img, h_img):
    boxes = []
    if not Path(lbl_path).exists(): return boxes
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5: continue
            cls_id = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:])
            x1 = (cx - bw/2)*w_img; y1 = (cy - bh/2)*h_img
            x2 = (cx + bw/2)*w_img; y2 = (cy + bh/2)*h_img
            boxes.append((cls_id, x1, y1, x2, y2))
    return boxes

IOU_THRESH  = 0.5
CONF_THRESH = 0.25
successes, failures = [], []

all_preds = model.predict(
    source  = [str(p) for p in test_images],
    imgsz   = IMG_SIZE,
    conf    = CONF_THRESH,
    iou     = 0.5,
    device  = DEVICE,
    save    = False,
    verbose = False,
)

for img_path, result in zip(test_images, all_preds):
    img = Image.open(img_path).convert("RGB")
    w_img, h_img = img.size
    gt_boxes = load_gt_boxes(TEST_LBL / (img_path.stem + ".txt"), w_img, h_img)
    if not gt_boxes: continue

    pred_boxes = []
    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            pred_boxes.append((int(box.cls[0].cpu()), float(box.conf[0].cpu()), x1, y1, x2, y2))

    matched = 0
    for gt in gt_boxes:
        gt_cls, gx1, gy1, gx2, gy2 = gt
        for pred in pred_boxes:
            p_cls, p_conf, px1, py1, px2, py2 = pred
            if p_cls == gt_cls and compute_iou([px1,py1,px2,py2],[gx1,gy1,gx2,gy2]) >= IOU_THRESH:
                matched += 1; break

    score = matched / len(gt_boxes)
    if score >= 1.0:
        successes.append((img_path, result, score))
    else:
        failures.append((img_path, result, score))

print(f"✅ Bien détectées (IoU≥{IOU_THRESH}) : {len(successes)}/{len(test_images)}")
print(f"❌ Avec erreurs                       : {len(failures)}/{len(test_images)}")

In [ ]:
def plot_cases(cases, title, n=6):
    random.shuffle(cases)
    sample_cases = cases[:n]
    cols = 3; rows = (len(sample_cases) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 5*rows))
    axes = axes.flatten() if rows*cols > 1 else [axes]
    cmap = plt.colormaps["tab10"]

    for ax, (img_path, result, score) in zip(axes, sample_cases):
        img = Image.open(img_path).convert("RGB")
        w_img, h_img = img.size
        ax.imshow(img)
        gt_boxes = load_gt_boxes(TEST_LBL / (img_path.stem + ".txt"), w_img, h_img)
        for (cls_id, x1, y1, x2, y2) in gt_boxes:
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor="lime", facecolor="none", linestyle="--"))
        if result.boxes is not None and len(result.boxes) > 0:
            for box in result.boxes:
                bx1, by1, bx2, by2 = box.xyxy[0].cpu().numpy()
                cls_id = int(box.cls[0].cpu()); conf = float(box.conf[0].cpu())
                color = cmap(cls_id / 10)
                ax.add_patch(patches.Rectangle((bx1, by1), bx2-bx1, by2-by1,
                    linewidth=2, edgecolor=color, facecolor="none"))
                ax.text(bx1, by1-4, f"{CLASS_NAMES[cls_id]} {conf:.2f}",
                        color="white", fontsize=8, fontweight="bold",
                        bbox=dict(facecolor=color, alpha=0.8, pad=1))
        ax.set_title(f"{img_path.name[:28]}\nscore={score:.2f}", fontsize=7)
        ax.axis("off")

    for ax in axes[len(sample_cases):]: ax.axis("off")
    legend_elements = [
        Line2D([0],[0], color="lime", linewidth=2, linestyle="--", label="Ground Truth"),
        Line2D([0],[0], color="gray", linewidth=2, linestyle="-",  label="Prédiction"),
    ]
    fig.legend(handles=legend_elements, loc="lower center", ncol=2, fontsize=10)
    plt.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    fname = title.lower().replace(" ","_").replace("(","").replace(")","") + ".png"
    plt.savefig(fname, dpi=150, bbox_inches="tight"); plt.show()
    print(f"Figure sauvegardée : {fname}")

plot_cases(successes, "Cas réussis (détections correctes)", n=6)
plot_cases(failures,  "Erreurs courantes du modèle", n=6)


### 4.3 Analyse des erreurs par classe

In [ ]:
class_errors = {name: {"tp": 0, "fp": 0, "fn": 0} for name in CLASS_NAMES}

for img_path, result in zip(test_images, all_preds):
    img = Image.open(img_path).convert("RGB")
    w_img, h_img = img.size
    gt_boxes = load_gt_boxes(TEST_LBL / (img_path.stem + ".txt"), w_img, h_img)
    pred_boxes = []
    if result.boxes is not None:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            pred_boxes.append((int(box.cls[0].cpu()), float(box.conf[0].cpu()), x1, y1, x2, y2))

    matched_gt = set()
    for pred in pred_boxes:
        p_cls, p_conf, px1, py1, px2, py2 = pred
        best_iou, best_gt_idx = 0, -1
        for gt_idx, (gt_cls, gx1, gy1, gx2, gy2) in enumerate(gt_boxes):
            if gt_idx in matched_gt or gt_cls != p_cls: continue
            iou = compute_iou([px1,py1,px2,py2],[gx1,gy1,gx2,gy2])
            if iou > best_iou: best_iou, best_gt_idx = iou, gt_idx
        if best_iou >= IOU_THRESH:
            class_errors[CLASS_NAMES[p_cls]]["tp"] += 1
            matched_gt.add(best_gt_idx)
        else:
            class_errors[CLASS_NAMES[p_cls]]["fp"] += 1
    for gt_idx, (gt_cls, *_) in enumerate(gt_boxes):
        if gt_idx not in matched_gt:
            class_errors[CLASS_NAMES[gt_cls]]["fn"] += 1

df_errors = pd.DataFrame(class_errors).T
df_errors["precision"] = df_errors["tp"] / (df_errors["tp"] + df_errors["fp"] + 1e-9)
df_errors["recall"]    = df_errors["tp"] / (df_errors["tp"] + df_errors["fn"] + 1e-9)
print("Analyse TP/FP/FN par classe :\n")
print(df_errors.to_string(float_format="{:.3f}".format))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(CLASS_NAMES)); width = 0.25

axes[0].bar(x-width, df_errors["tp"], width, label="TP", color="#4CAF50", edgecolor="black")
axes[0].bar(x,       df_errors["fp"], width, label="FP", color="#F44336", edgecolor="black")
axes[0].bar(x+width, df_errors["fn"], width, label="FN", color="#FF9800", edgecolor="black")
axes[0].set_xticks(x); axes[0].set_xticklabels(CLASS_NAMES)
axes[0].set_title("TP / FP / FN par classe"); axes[0].legend()

axes[1].bar(x-width/2, df_errors["precision"], width, label="Précision", color="#2196F3", edgecolor="black")
axes[1].bar(x+width/2, df_errors["recall"],    width, label="Rappel",    color="#9C27B0", edgecolor="black")
axes[1].set_xticks(x); axes[1].set_xticklabels(CLASS_NAMES)
axes[1].set_title("Précision / Rappel par classe")
axes[1].set_ylim(0, 1.05); axes[1].legend()

plt.suptitle("Analyse des erreurs par classe — Jeu de test", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("error_analysis.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Résumé Final

In [ ]:
print("="*60)
print("RÉSUMÉ FINAL — ÉVALUATION SUR LE JEU DE TEST")
print("="*60)
print(f"  Modèle         : {BEST_PT.name}")
print(f"  Architecture   : yolov8s (small) — Transfer Learning COCO")
print(f"  Images testées : {len(test_images)}")
print(f"  Classes        : {CLASS_NAMES}")
print()
print("  Métriques globales :")
print(f"    mAP@50      : {map50:.4f}")
print(f"    mAP@50-95   : {map5095:.4f}")
print(f"    Précision   : {prec:.4f}")
print(f"    Rappel      : {rec:.4f}")
print(f"    F1-score    : {f1:.4f}")
print()
print("  Taux de succès (IoU≥0.5) :")
print(f"    Bien détectées : {len(successes)}/{len(test_images)} ({100*len(successes)/len(test_images):.1f}%)")
print("="*60)

## Conclusion

Ce notebook a évalué le meilleur modèle YOLOv8 (yolov8s, lr=5e-4, augmentation renforcée)
sur le **jeu de test** (25 images, jamais vues pendant l'entraînement).

Les points clés :
- Métriques quantitatives globales et par classe (mAP@50, mAP@50-95, précision, rappel)
- Visualisation des prédictions vs ground truth sur 9 images
- Identification des cas réussis et des erreurs typiques
- Analyse TP/FP/FN par classe pour cibler les personnages difficiles